In [23]:
from pathlib import Path
import random
print(Path.cwd())


d:\虚拟C盘\大学\大学项目\用Transformer实现机器翻译\data


In [24]:
# 获取当前脚本所在的文件夹
folder = Path.cwd()

# 找到中文文件和英文文件
zh_path = folder/"UNPC.en-zh.zh"
en_path = folder/"UNPC.en-zh.en"
  
with zh_path.open("r", encoding="utf-8") as zh_file, \
     en_path.open("r", encoding="utf-8") as en_file:
    count_row_en = sum(1 for _ in en_file)
    count_row_zh = sum(1 for _ in zh_file)
    print(f"中文文件总行数：{count_row_zh}")
    print(f"英文文件总行数：{count_row_en}")

中文文件总行数：17451549
英文文件总行数：17451549


In [25]:
sample_num = 20000
random.seed(22)
sample_indices = set(random.sample(range(count_row_zh),sample_num))
output_path_zh = folder/"sample_zh.txt"
output_path_en = folder/"sample_en.txt"
with output_path_zh.open("w",encoding="utf-8") as f,\
    output_path_en.open("w",encoding='utf-8') as g,\
     en_path.open("r",encoding="utf-8") as en_file,\
     zh_path.open("r",encoding="utf-8") as zh_file:
    for i, (line1,line2) in enumerate(zip(zh_file,en_file)):
        if i in sample_indices:
            line1 = line1.rstrip("\r\n")
            line2 = line2.rstrip("\r\n")
            f.write(line1 + "\n")
            g.write(line2 + "\n")


In [26]:
with output_path_zh.open("r", encoding="utf-8") as zh_file, \
     output_path_en.open("r", encoding="utf-8") as en_file:
    count_row_output_en = sum(1 for _ in en_file)
    count_row_output_zh = sum(1 for _ in zh_file)
    print(f"中文文件总行数：{count_row_output_zh}")
    print(f"英文文件总行数：{count_row_output_en}")

中文文件总行数：20000
英文文件总行数：20000


In [27]:
with output_path_zh.open("r", encoding="utf-8") as zh_file, \
     output_path_en.open("r", encoding="utf-8") as en_file:
    sample_list = [ count_row_output_zh//4 * i for i in range(5)]
    for i, (zh, en) in enumerate(zip(zh_file, en_file)):
        if i in sample_list:
            print(f"\n第{i + 1}行,中文：{zh.rstrip()}, 英文：{en.rstrip()}")



第1行,中文：23. 中欧高级研究卫星(CESAR)是一个重约300公斤的航天器，它将在一条近地点为400公里、远地点为1,000公里、倾角为70度的轨道上飞行。, 英文：The Central European Satellite for Advanced Research (CESAR) is a spacecraft of about 300 kg that will fly in an orbit with a perigee of 400 km, an apogee of 1,000 km and an inclination of 70 degrees.

第5001行,中文：8. 国际法官协会, 英文：International Association of Judges

第10001行,中文：报告突出说明了非洲国家在加强其统计能力方面面临的问题和挑战。, 英文：The report highlights issues and challenges facing African countries in enhancing their statistical capacity.

第15001行,中文：因此，建议在其他地方述及脚注3中提出的问题。, 英文：The suggestion was therefore made that the issues raised in footnote 3 should be addressed elsewhere.


##### 划分数据集为训练集、验证集和测试集

In [28]:
pairs = []
with output_path_en.open("r", encoding="utf-8") as en_file,\
     output_path_zh.open("r", encoding="utf-8") as zh_file:
    for i, (en,zh ) in enumerate(zip(en_file, zh_file)):
        pairs.append((en.rstrip("\r\n"), zh.rstrip("\r\n")))
random.Random(22).shuffle(pairs)

storage_dir = folder/"sample_for_train_val_test"
storage_dir.mkdir(parents=True,exist_ok=True)

output_train_zh = storage_dir/"train_zh.txt"
output_train_en = storage_dir/"train_en.txt"
output_val_zh = storage_dir/"val_zh.txt"
output_val_en = storage_dir/"val_en.txt"
output_test_zh = storage_dir/"test_zh.txt"
output_test_en = storage_dir/"test_en.txt"

with output_train_zh.open("w", encoding="utf-8") as train_zh_file, \
     output_train_en.open("w", encoding="utf-8") as train_en_file, \
     output_val_zh.open("w", encoding="utf-8") as val_zh_file, \
     output_val_en.open("w", encoding="utf-8") as val_en_file, \
     output_test_zh.open("w", encoding="utf-8") as test_zh_file, \
     output_test_en.open("w", encoding="utf-8") as test_en_file:
    for i,item in enumerate(pairs):
        if i <int(0.9*len(pairs)):
            train_en_file.write(item[0] + "\n")
            train_zh_file.write(item[1] + "\n")
        elif i<int(0.95*len(pairs)):
            val_en_file.write(item[0] + "\n")
            val_zh_file.write(item[1] + "\n")
        else:
            test_en_file.write(item[0] + "\n")
            test_zh_file.write(item[1] + "\n")